In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

import tensorflow as tf
import keras_tuner as kt
import tensorboard
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout,BatchNormalization,Input,LSTM,Bidirectional
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error,r2_score

In [3]:
df_train = pd.read_csv(r'../data/cleaned/train.csv')
df_test = pd.read_csv(r'../data/cleaned/test.csv')
df_val = pd.read_csv(r'../data/cleaned/val.csv')

In [4]:
with open('target_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [5]:
def create_sequences(df,seq_length, target_col='PJME_MW',horizon=1):
    """
    Converts a scaled DataFrame into 3D sequence arrays for Keras models.
    """
    # Separate feature columns from Datetime
    feature_cols = [col for col in df.columns if col != 'Datetime']
    target_idx = feature_cols.index(target_col)
    
    data = df[feature_cols].values
    X, y = [], []
    
    for i in range(len(data) - seq_length - horizon + 1):
        X.append(data[i : i + seq_length, :])
        if horizon == 1:
            y.append(data[i + seq_length, target_idx])
        else:
            y.append(data[i + seq_length : i + seq_length + horizon, target_idx])
            
    return np.array(X), np.array(y)

In [6]:
SEQ_LEN = 168
HORIZON = 24

X_train168, y_train168 = create_sequences(df_train, seq_length=SEQ_LEN, horizon=HORIZON)
X_val168, y_val168     = create_sequences(df_val, seq_length=SEQ_LEN, horizon=HORIZON)
X_test168, y_test168   = create_sequences(df_test, seq_length=SEQ_LEN, horizon=HORIZON)

print("X_train:", X_train168.shape)
print("y_train:", y_train168.shape)

print("X_test:", X_test168.shape)
print("y_test:", y_test168.shape)

X_train: (101449, 168, 13)
y_train: (101449, 24)
X_test: (21589, 168, 13)
y_test: (21589, 24)


In [7]:
y_test_mw = scaler.inverse_transform(y_test168.reshape(-1, 1))

In [8]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

In [9]:
early_stop = EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True,verbose=1)

In [ ]:
#drop out 0.2

In [10]:
model_bilstm = Sequential([
    Bidirectional(LSTM(64, return_sequences=False), input_shape=(X_train168.shape[1], X_train168.shape[2])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_bilstm.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_bilstm = model_bilstm.fit(
    X_train168, y_train168,
    validation_data=(X_val168, y_val168),
    epochs=10,
    batch_size=64,
    verbose=1
)

E0000 00:00:1786447609.561290  741653 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 122s 76ms/step - loss: 0.0057 - mae: 0.0526 - val_loss: 0.0020 - val_mae: 0.0335
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 130s 82ms/step - loss: 0.0020 - mae: 0.0331 - val_loss: 0.0019 - val_mae: 0.0313
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 126s 79ms/step - loss: 0.0017 - mae: 0.0308 - val_loss: 0.0018 - val_mae: 0.0303
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 137s 86ms/step - loss: 0.0016 - mae: 0.0298 - val_loss: 0.0017 - val_mae: 0.0300
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 126s 80ms/step - loss: 0.0016 - mae: 0.0291 - val_loss: 0.0016 - val_mae: 0.0282
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 127s 80ms/step - loss: 0.0015 - mae: 0.0285 - val_loss: 0.0016 - val_mae: 0.0286
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 124s 78ms/step - loss: 0.0015 - mae: 0.0283 - val_loss: 0.0016 - val_mae: 0.0287
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 119s 75ms/step - loss: 0.0015 - mae: 0.0279 - val_loss: 0.0016 - val_mae: 0.0287
Epoch 9/10
1586/

In [11]:
y_pred_bilstm_scaled = model_bilstm.predict(X_test168)

y_pred_bilstm_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_bilstm  = mean_absolute_error(y_test_mw, y_pred_bilstm_mw)
rmse_bilstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_bilstm_mw))
mape_bilstm = np.mean(np.abs((y_test_mw - y_pred_bilstm_mw) / y_test_mw)) * 100
r2_bilstm  = r2_score(y_test_mw, y_pred_bilstm_mw)

print(f"MAE:  {mae_bilstm:.2f} MW")
print(f"RMSE: {rmse_bilstm:.2f} MW")
print(f"MAPE: {mape_bilstm:.2f}%")
print(f"R2:   {r2_bilstm:.4f}")

675/675 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step
MAE:  1483.72 MW
RMSE: 2104.47 MW
MAPE: 4.57%
R2:   0.8932


In [ ]:
#0.3

In [12]:
model_bilstm = Sequential([
    Bidirectional(LSTM(64, return_sequences=False), input_shape=(X_train168.shape[1], X_train168.shape[2])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_bilstm.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_bilstm = model_bilstm.fit(
    X_train168, y_train168,
    validation_data=(X_val168, y_val168),
    epochs=10,
    batch_size=64,
    verbose=1
)

/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 126s 78ms/step - loss: 0.0064 - mae: 0.0565 - val_loss: 0.0020 - val_mae: 0.0330
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 128s 81ms/step - loss: 0.0021 - mae: 0.0346 - val_loss: 0.0018 - val_mae: 0.0317
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 117s 74ms/step - loss: 0.0019 - mae: 0.0325 - val_loss: 0.0017 - val_mae: 0.0299
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 119s 75ms/step - loss: 0.0018 - mae: 0.0316 - val_loss: 0.0016 - val_mae: 0.0293
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 128s 81ms/step - loss: 0.0017 - mae: 0.0308 - val_loss: 0.0019 - val_mae: 0.0309
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 118s 74ms/step - loss: 0.0017 - mae: 0.0304 - val_loss: 0.0020 - val_mae: 0.0330
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 131s 83ms/step - loss: 0.0016 - mae: 0.0299 - val_loss: 0.0015 - val_mae: 0.0281
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 133s 84ms/step - loss: 0.0016 - mae: 0.0295 - val_loss: 0.0015 - val_mae: 0.0286
Epoch 9/10
1586/

In [13]:
y_pred_bilstm_scaled = model_bilstm.predict(X_test168)

y_pred_bilstm_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_bilstm  = mean_absolute_error(y_test_mw, y_pred_bilstm_mw)
rmse_bilstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_bilstm_mw))
mape_bilstm = np.mean(np.abs((y_test_mw - y_pred_bilstm_mw) / y_test_mw)) * 100
r2_bilstm  = r2_score(y_test_mw, y_pred_bilstm_mw)

print(f"MAE:  {mae_bilstm:.2f} MW")
print(f"RMSE: {rmse_bilstm:.2f} MW")
print(f"MAPE: {mape_bilstm:.2f}%")
print(f"R2:   {r2_bilstm:.4f}")

675/675 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step
MAE:  1469.83 MW
RMSE: 2007.76 MW
MAPE: 4.66%
R2:   0.9028


In [14]:
model_bilstm.save(r'../models/drop_bilstm.keras')
history_df =pd.DataFrame(history_bilstm.history)

history_df.to_csv(r'../log/drop_bilstm.csv',index=False)

In [ ]:
#0.5

In [15]:
model_bilstm = Sequential([
    Bidirectional(LSTM(64, return_sequences=False), input_shape=(X_train168.shape[1], X_train168.shape[2])),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_bilstm.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_bilstm = model_bilstm.fit(
    X_train168, y_train168,
    validation_data=(X_val168, y_val168),
    epochs=10,
    batch_size=64,
    verbose=1
)

/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 123s 76ms/step - loss: 0.0079 - mae: 0.0622 - val_loss: 0.0025 - val_mae: 0.0380
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 128s 81ms/step - loss: 0.0027 - mae: 0.0394 - val_loss: 0.0023 - val_mae: 0.0373
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 123s 77ms/step - loss: 0.0023 - mae: 0.0362 - val_loss: 0.0018 - val_mae: 0.0316
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 122s 77ms/step - loss: 0.0022 - mae: 0.0350 - val_loss: 0.0018 - val_mae: 0.0314
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 122s 77ms/step - loss: 0.0021 - mae: 0.0343 - val_loss: 0.0017 - val_mae: 0.0299
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 120s 76ms/step - loss: 0.0020 - mae: 0.0337 - val_loss: 0.0017 - val_mae: 0.0300
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 128s 81ms/step - loss: 0.0020 - mae: 0.0331 - val_loss: 0.0016 - val_mae: 0.0295
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 135s 85ms/step - loss: 0.0019 - mae: 0.0326 - val_loss: 0.0016 - val_mae: 0.0291
Epoch 9/10
1586/

In [16]:
y_pred_bilstm_scaled = model_bilstm.predict(X_test168)

y_pred_bilstm_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_bilstm  = mean_absolute_error(y_test_mw, y_pred_bilstm_mw)
rmse_bilstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_bilstm_mw))
mape_bilstm = np.mean(np.abs((y_test_mw - y_pred_bilstm_mw) / y_test_mw)) * 100
r2_bilstm  = r2_score(y_test_mw, y_pred_bilstm_mw)

print(f"MAE:  {mae_bilstm:.2f} MW")
print(f"RMSE: {rmse_bilstm:.2f} MW")
print(f"MAPE: {mape_bilstm:.2f}%")
print(f"R2:   {r2_bilstm:.4f}")

675/675 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step
MAE:  1500.72 MW
RMSE: 2065.52 MW
MAPE: 4.72%
R2:   0.8971
